for each pathway change it in cell 4 and then run cell 6

In [6]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import warnings
from pathlib import Path

import numpy as np
import joblib

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=RuntimeWarning)
print('Imports OK')

Imports OK


In [7]:
# ── Cell 2: Paths ─────────────────────────────────────────────────────────────
_BASE = (
    '/Users/farjam/Library/CloudStorage'
    '/OneDrive-UniversityofEdinburgh/wellcome/Amir/Final_VEP'
)
DATA_NPZ_ROOT = Path(_BASE) / 'DATA_NPZ'
ECOC_ROOT     = Path(_BASE) / 'ECOC'

ECOC_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Data : {DATA_NPZ_ROOT}')
print(f'ECOC : {ECOC_ROOT}')

Data : /Users/farjam/Library/CloudStorage/OneDrive-UniversityofEdinburgh/wellcome/Amir/Final_VEP/DATA_NPZ
ECOC : /Users/farjam/Library/CloudStorage/OneDrive-UniversityofEdinburgh/wellcome/Amir/Final_VEP/ECOC


In [8]:
# ── Cell 3: Subjects ──────────────────────────────────────────────────────────
AGE_DATA = {
    '1001': 25.0, '1002': 43.0, '1004': 33.0, '1005': 50.0, '1007': 31.0,
    '1008': 25.0, '1010': 26.0, '1011': 37.0, '1014': 23.0, '1016': 36.0,
    '1017': 46.0, '1018': 25.0, '1020': 29.0, '1021': 28.0, '1023': 46.0,
    '1024': 24.0, '1026': 25.0, '1028': 62.0, '1034': 60.0, '1038': 45.0,
    '1039': 63.0, '1041': 49.0, '1042': 39.0, '1044': 62.0, '1046': 61.0,
    '1052': 53.0, '2002': 62.0, '2006': 25.0, '2009': 49.0, '2017': 31.0,
    '2020': 43.0, '2023': 49.0, '2026': 47.0, '2028': 29.0, '2029': 50.0,
    '2037': 45.0, '3001': 54.0, '3005': 39.0, '3006': 33.0, '3007': 41.0,
    '3008': 32.0, '3011': 31.0, '3014': 34.0, '3016': 44.0, '3027': 72.0,
    '3030': 54.0, '3034': 48.0, '3039': 34.0, '3041': 35.0,
}
SUBJECTS = list(AGE_DATA.keys())
print(f'{len(SUBJECTS)} subjects loaded')

49 subjects loaded


In [14]:
# ── Cell 4: Pathway & decoding parameters ────────────────────────────────────

# ← Change this to 'Luminance', 'L-M', or 'S-cone'
PATHWAY = 'S-cone'

PATHWAYS = {
    'Luminance': {
        'contrasts': np.array([0.035, 0.07,  0.14,  0.28],  float),
        'folder':    'Luminance_NPZ',
    },
    'L-M': {
        'contrasts': np.array([0.008, 0.016, 0.032, 0.064], float),
        'folder':    'L_M_NPZ',
    },
    'S-cone': {
        'contrasts': np.array([0.06,  0.12,  0.24,  0.48],  float),
        'folder':    'S_cone_NPZ',
    },
}

WINDOW_SIZE   = 5     # samples (~20 ms at 256 Hz)
TIME_RANGE    = [0.05, 0.30]  # seconds — used to find peak window
N_SPLITS      = 4     # stratified k-fold
N_REPEATS     = 10    # repeated CV

# Derived from PATHWAY choice
contrasts  = PATHWAYS[PATHWAY]['contrasts']
n_levels   = len(contrasts)
data_dir   = DATA_NPZ_ROOT / PATHWAYS[PATHWAY]['folder']
ecoc_dir   = ECOC_ROOT / PATHWAY
ecoc_dir.mkdir(parents=True, exist_ok=True)

print(f'Pathway   : {PATHWAY}')
print(f'Contrasts : {contrasts}')
print(f'Data dir  : {data_dir}')
print(f'ECOC dir  : {ecoc_dir}')

Pathway   : S-cone
Contrasts : [0.06 0.12 0.24 0.48]
Data dir  : /Users/farjam/Library/CloudStorage/OneDrive-UniversityofEdinburgh/wellcome/Amir/Final_VEP/DATA_NPZ/S_cone_NPZ
ECOC dir  : /Users/farjam/Library/CloudStorage/OneDrive-UniversityofEdinburgh/wellcome/Amir/Final_VEP/ECOC/S-cone


In [10]:
# ── Cell 5: Helper functions ──────────────────────────────────────────────────

def load_subject(subject):
    """Load NPZ → X (trials, 64, time), y (labels), times (s)."""
    fp = data_dir / f'sub-{subject}_{PATHWAY}_data.npz'
    with np.load(fp, allow_pickle=False) as d:
        X     = d['X'].astype(np.float32, copy=False)
        y     = d['y'].ravel().astype(np.int32, copy=False)
        times = d['times'].ravel().astype(np.float64, copy=False)
    # Tolerate transposed layout
    if X.shape[-1] != times.size and X.shape[1] == times.size:
        X = np.transpose(X, (0, 2, 1))
    assert X.shape[-1] == times.size, f'time mismatch sub-{subject}'
    return X, y, times


def peak_window(t, acc):
    """Index of peak accuracy inside TIME_RANGE, fallback to global peak."""
    inr = (t >= TIME_RANGE[0]) & (t <= TIME_RANGE[1])
    if np.any(inr):
        return int(np.where(inr)[0][np.argmax(acc[inr])])
    return int(np.argmax(acc))


def make_clf():
    """LDA with Ledoit-Wolf shrinkage — standard for EEG decoding."""
    return make_pipeline(
        StandardScaler(),
        LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'),
    )


print('Helpers OK')

Helpers OK


In [15]:
# ── Cell 6: Decoding loop ─────────────────────────────────────────────────────
#
# Saves per subject:
#
#   sub-<ID>_ecoc_results.npz
#       time_decoder       (n_windows,)          window centre times in seconds
#       accuracies         (n_windows,)          mean CV accuracy
#       confusion_matrices (n_windows, 4, 4)     row-normalised
#
#   sub-<ID>_<PATHWAY>_clf.joblib
#       classifier fitted on ALL trials at the subject's peak window
#       keys: clf, subject, pathway, peak_time_s, contrasts
#
# Skips subjects whose files already exist — safe to re-run.

for subject in SUBJECTS:

    out_npz = ecoc_dir / f'sub-{subject}_ecoc_results.npz'
    out_clf = ecoc_dir / f'sub-{subject}_{PATHWAY}_clf.joblib'

    if out_npz.exists() and out_clf.exists():
        print(f'[{subject}] already done — skip')
        continue

    # Load data
    try:
        X, y, times = load_subject(subject)
    except FileNotFoundError:
        print(f'[{subject}] NPZ not found — skip')
        continue

    _, _, n_t = X.shape
    n_win     = n_t - WINDOW_SIZE + 1
    t_win     = times[WINDOW_SIZE // 2 : WINDOW_SIZE // 2 + n_win]

    accs = np.zeros(n_win)
    cms  = np.zeros((n_win, n_levels, n_levels))
    clf  = make_clf()

    # Sliding window
    for w in range(n_win):
        Xw     = X[:, :, w : w + WINDOW_SIZE].mean(axis=2)  # (trials, channels)
        cm_sum = np.zeros((n_levels, n_levels))
        fa     = []

        for rep in range(N_REPEATS):
            cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=rep)
            for tr, te in cv.split(Xw, y):
                clf.fit(Xw[tr], y[tr])
                yp = clf.predict(Xw[te])
                fa.append((yp == y[te]).mean())
                cm_sum += confusion_matrix(y[te], yp, labels=np.arange(n_levels))

        accs[w] = np.mean(fa)
        row = cm_sum.sum(axis=1, keepdims=True)
        row[row == 0] = 1.0
        cms[w] = cm_sum / row

    # Save results
    np.savez(out_npz,
             time_decoder=t_win,
             accuracies=accs,
             confusion_matrices=cms)

    # Save classifier at peak window (fitted on all trials)
    pidx   = peak_window(t_win, accs)
    Xw_pk  = X[:, :, pidx : pidx + WINDOW_SIZE].mean(axis=2)
    clf_pk = make_clf()
    clf_pk.fit(Xw_pk, y)
    joblib.dump({
        'clf':        clf_pk,
        'subject':    subject,
        'pathway':    PATHWAY,
        'peak_time_s': float(t_win[pidx]),
        'contrasts':  contrasts,
    }, out_clf)

    print(f'[{subject}] saved | windows={n_win} | peak={t_win[pidx]:.3f}s')

[1001] saved | windows=252 | peak=0.297s
[1002] saved | windows=252 | peak=0.137s
[1004] saved | windows=252 | peak=0.219s
[1005] saved | windows=252 | peak=0.145s
[1007] saved | windows=252 | peak=0.230s
[1008] saved | windows=252 | peak=0.258s
[1010] saved | windows=252 | peak=0.211s
[1011] saved | windows=252 | peak=0.137s
[1014] saved | windows=252 | peak=0.125s
[1016] saved | windows=252 | peak=0.168s
[1017] saved | windows=252 | peak=0.180s
[1018] saved | windows=252 | peak=0.215s
[1020] saved | windows=252 | peak=0.148s
[1021] saved | windows=252 | peak=0.156s
[1023] saved | windows=252 | peak=0.297s
[1024] saved | windows=252 | peak=0.141s
[1026] saved | windows=252 | peak=0.133s
[1028] saved | windows=252 | peak=0.203s
[1034] saved | windows=252 | peak=0.270s
[1038] saved | windows=252 | peak=0.234s
[1039] saved | windows=252 | peak=0.195s
[1041] saved | windows=252 | peak=0.152s
[1042] saved | windows=252 | peak=0.125s
[1044] saved | windows=252 | peak=0.172s
[1046] saved | w